In [0]:
# ============================================================
# CELL 1: Import Required Libraries
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as spark_sum

# Check that imports are working
print("Imports completed successfully")

Imports completed successfully


In [0]:
# ============================================================
# CELL 2: Configure Azure ADLS Gen2 Storage
# ============================================================

from pyspark.sql import functions as F

storage_account = "healthcarestoragerev01"
container = "input"

storage_key = dbutils.secrets.get(
    scope="healthcare-scope",
    key="storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

# Define Gold Layer path
gold_path = (
    f"abfss://input@{storage_account}.dfs.core.windows.net/gold"
)

print("Storage Account:", storage_account)
print("Gold Path:", gold_path)
print("Storage authentication configured successfully")

Storage Account: healthcarestoragerev01
Gold Path: abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold
Storage authentication configured successfully


In [0]:
# ============================================================
# CELL 3: Load Provider and Transaction Gold Data
# ============================================================

# Define Gold table paths
provider_path = f"{gold_path}/providers"
transaction_path = f"{gold_path}/transactions"

# Read Provider Gold table
provider_df = (
    spark.read
    .format("delta")
    .load(provider_path)
)

# Read Transaction Gold table
transaction_df = (
    spark.read
    .format("delta")
    .load(transaction_path)
)

# Display record counts
print("Provider records:", provider_df.count())
print("Transaction records:", transaction_df.count())

Provider records: 25
Transaction records: 10000


In [0]:
# ============================================================
# CELL 4: Check Provider and Transaction Columns
# ============================================================

print("Provider columns:")
print(provider_df.columns)

print("\nTransaction columns:")
print(transaction_df.columns)

Provider columns:
['ProviderID', 'FirstName', 'LastName', 'Specialization', 'DeptID', 'NPI', '_bronze_loaded_at', '_silver_load_timestamp']

Transaction columns:
['TransactionID', 'EncounterID', 'PatientID', 'ProviderID', 'DeptID', 'VisitDate', 'ServiceDate', 'PaidDate', 'VisitType', 'Amount', 'AmountType', 'PaidAmount', 'ClaimID', 'PayorID', 'ProcedureCode', 'ICDCode', 'LineOfBusiness', 'MedicaidID', 'MedicareID', 'InsertDate', 'ModifiedDate', '_bronze_loaded_at', '_silver_load_timestamp']


In [0]:
# ============================================================
# CELL 5: Calculate Provider Revenue
# ============================================================

from pyspark.sql.functions import col, sum as spark_sum

provider_revenue_df = (
    transaction_df
    .groupBy("ProviderID")
    .agg(
        spark_sum("PaidAmount").alias("TotalRevenue")
    )
)

print("Provider Revenue calculated successfully")
display(provider_revenue_df.limit(20))

Provider Revenue calculated successfully


ProviderID,TotalRevenue
PROV0231,9397.900016784668
PROV0485,5572.129989624023
PROV0139,5388.0999755859375
PROV0369,5204.29997253418
PROV0493,12193.289901733398
PROV0129,6014.129997253418
PROV0364,9277.730117797852
PROV0288,11333.839988708496
PROV0425,7181.660037994385
PROV0049,8404.629928588867


In [0]:
# ============================================================
# CELL 6: Verify Provider Revenue
# ============================================================

print(
    "Provider Revenue records:",
    provider_revenue_df.count()
)

display(
    provider_revenue_df.limit(20)
)

Provider Revenue records: 500


ProviderID,TotalRevenue
PROV0231,9397.900016784668
PROV0485,5572.129989624023
PROV0139,5388.0999755859375
PROV0369,5204.29997253418
PROV0493,12193.289901733398
PROV0129,6014.129997253418
PROV0364,9277.730117797852
PROV0288,11333.839988708496
PROV0425,7181.660037994385
PROV0049,8404.629928588867


In [0]:
# ============================================================
# CELL 7: Save Provider Revenue to Gold Layer
# ============================================================

# Define output path
provider_revenue_path = (
    f"{gold_path}/provider_revenue"
)

# Save as Delta
(
    provider_revenue_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(provider_revenue_path)
)

print("Provider Revenue saved successfully")
print("Path:", provider_revenue_path)

Provider Revenue saved successfully
Path: abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/provider_revenue


In [0]:
# ============================================================
# CELL 8: Final Verification
# ============================================================

# Read saved Provider Revenue table
final_provider_revenue_df = (
    spark.read
    .format("delta")
    .load(provider_revenue_path)
)

# Display record count
print(
    "Provider Revenue records:",
    final_provider_revenue_df.count()
)

# Display final result
display(
    final_provider_revenue_df
    .orderBy(
        col("TotalRevenue").desc()
    )
)

print(
    "Provider Revenue verification completed successfully"
)

Provider Revenue records: 500


ProviderID,TotalRevenue
PROV0041,14807.98006439209
PROV0476,14456.709980010986
PROV0080,14418.240036010742
PROV0155,14287.55011177063
PROV0158,14184.379825592041
PROV0475,14055.639957427979
PROV0217,13849.87009048462
PROV0098,13544.489944458008
PROV0124,13370.169933319092
PROV0495,13168.999969482422


Provider Revenue verification completed successfully
